# 05 - reasoning-model corpus and analysis\n\nGenerate the Qwen3 reasoning corpus, tidy the repetition (Arm B), and check whether structure predicts correctness. Long run: save to Drive, it is resumable.

### setup

In [ ]:
import os, sys, subprocess
REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"
if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)
subprocess.run(["git", "fetch", "-q"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)
sys.path.insert(0, os.getcwd())
print("ok", os.getcwd())

### deps

In [ ]:
!pip install -q transformers accelerate bitsandbytes datasets networkx pyyaml

In [ ]:
import torch; print("gpu:", torch.cuda.is_available())

### save to drive so a long run survives a disconnect

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUT = "/content/drive/MyDrive/project-rl-arg/corpus_reasoning.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)

### quick test - 5 traces first (check <think> tags + answers look right)

In [ ]:
from e0_corpus import generate, summarise
generate(n_questions=5, out=OUT, max_new_tokens=8192, fresh=True)

### the real run - 300 traces (several hours at 8192).\nResumable: if Colab drops, re-run this **without** fresh=True to continue.

In [ ]:
generate(n_questions=300, out=OUT, max_new_tokens=8192, fresh=True)

### tidy the repetition (Arm B) and re-score

In [ ]:
from e0_corpus import rescore
DEDUP = OUT.replace(".jsonl", "_dedup.jsonl")
rescore(OUT, DEDUP, dedupe=True)

### compare raw (Arm A) vs tidied (Arm B)

In [ ]:
summarise(OUT)      # raw
summarise(DEDUP)    # tidied

### does structure predict correctness? the AUC on the reasoning model

In [ ]:
from analyse import run
run(OUT)
run(DEDUP)

### hand-check a few traces before trusting anything

In [ ]:
from e0_corpus import inspect
inspect(OUT, only="wrong", family="web_of_lies", n=3)